# Homework 12: Results Reporting and Delivery Design

Generates the charts and sensitivity table behind `reports/final_report.md`, a written-report
deliverable on a synthetic three-scenario portfolio analysis (baseline, alternate imputation,
alternate outlier rule).

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120
np.random.seed(101)

## Data
Three portfolio-construction scenarios, plus a monthly metric by asset class for the time-series chart.

In [2]:
processed_dir = Path('data/processed')
processed_dir.mkdir(parents=True, exist_ok=True)

df = pd.DataFrame({
    'scenario': ['baseline', 'alt_impute', 'alt_outlier'],
    'return': [0.12, 0.10, 0.14],
    'volatility': [0.18, 0.185, 0.19],
    'sharpe': [0.56, 0.43, 0.63],
    'assumption': ['imputation', 'imputation', 'outlier_rule'],
    'value': ['median_fill', 'mean_fill', '3sigma_clip'],
})
df.to_csv(processed_dir / 'final_results.csv', index=False)

months = pd.date_range('2025-01-01', periods=6, freq='MS')
ts = pd.DataFrame({
    'Date': list(months) * 3,
    'Category': ['Equities'] * 6 + ['Bonds'] * 6 + ['Commodities'] * 6,
    'MetricA': np.concatenate([
        75 + np.cumsum(np.random.normal(1.5, 3, 6)),
        60 + np.cumsum(np.random.normal(0.3, 1.5, 6)),
        90 + np.cumsum(np.random.normal(-0.5, 4, 6)),
    ]),
})
df.head()

,scenario,return,volatility,sharpe,assumption,value
0,baseline,0.12,0.180,0.56,imputation,median_fill
1,alt_impute,0.10,0.185,0.43,imputation,mean_fill
2,alt_outlier,0.14,0.190,0.63,outlier_rule,3sigma_clip


## Export Directory

In [3]:
img_dir = Path('reports/images')
img_dir.mkdir(parents=True, exist_ok=True)

def savefig(name):
    plt.tight_layout()
    plt.savefig(img_dir / name, dpi=200)
    plt.close()
    print(f'Saved {img_dir / name}')

## Chart 1: Risk-Return Scatter

In [4]:
plt.figure(figsize=(7, 5))
sns.scatterplot(data=df, x='volatility', y='return', hue='scenario', s=120)
plt.title('Risk-Return by Scenario')
plt.xlabel('Volatility')
plt.ylabel('Return')
savefig('risk_return.png')

Saved reports\images\risk_return.png


## Chart 2: Return by Scenario

In [5]:
plt.figure(figsize=(7, 5))
sns.barplot(data=df, x='scenario', y='return', hue='scenario', legend=False)
plt.title('Return by Scenario')
plt.ylabel('Return')
savefig('return_by_scenario.png')

Saved reports\images\return_by_scenario.png


## Chart 3: MetricA Over Time by Asset Class

In [6]:
plt.figure(figsize=(7, 5))
sns.lineplot(data=ts, x='Date', y='MetricA', hue='Category', marker='o')
plt.title('MetricA Over Time by Asset Class')
plt.xlabel('Date')
plt.ylabel('MetricA')
savefig('metricA_over_time.png')

Saved reports\images\metricA_over_time.png


## Sensitivity Table

In [7]:
baseline_return = df.loc[df['scenario'] == 'baseline', 'return'].iloc[0]
sensitivity = df[df['scenario'] != 'baseline'].copy()
sensitivity['delta_return'] = sensitivity['return'] - baseline_return
sensitivity[['scenario', 'assumption', 'value', 'return', 'delta_return']]

,scenario,assumption,value,return,delta_return
1,alt_impute,imputation,mean_fill,0.10,-0.02
2,alt_outlier,outlier_rule,3sigma_clip,0.14,0.02
